In [ ]:
#these are the liberaies I used for the whole project, but for preprocessing not all of them are used
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import numpy as np
import os
#from aerosound.params import *
from keras import Sequential, layers
from keras.metrics import Recall
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

## Convert .wav to .csv

In [2]:
# change to the correct folder in order to be able to import the following package
print(os.getcwd())
os.chdir("../..")
print(os.getcwd())

/Users/shaotongwang/code/Marie-Jourdain/aerosound/aerosound/model
/Users/shaotongwang/code/Marie-Jourdain/aerosound


In [3]:
from aerosound.ml_logic.data import *
from aerosound.ml_logic.preprocessing import *
# I updated the preprocessing.py, so please use the pull the preprocessing.py in my branch

In [ ]:
def get_data_rnn(path):
# this function is used to convert many .wav files in folder with subfolders to a dictionary of dataframes
    # 1, obtain data linked with the given path,
    # 2, convert it from .wav to dataframe,
    # 3, saperated the data by machine type (key) and save the dataframes (value) into a dictionary,
    # 4, return to this dictionary
    audio_paths = list(Path(path).rglob("*.wav"))
    df = build_dataset(audio_paths)
    machine_dfs = {}

    for machine in df['machine'].unique():
        machine_dfs[machine] = df[
            df['machine'] == machine
        ].copy()

    print(machine_dfs.keys())
    return machine_dfs

In [5]:
def get_dataframe(machine_dfs, machine_name):
    #obtain the dataframe of the certain machin
    return machine_dfs[machine_name]

In [ ]:
def get_data_single_wav(path):
# this is the function that used for API to convert one .wav to dataframe
    audio_paths = list(Path(path).rglob("*.wav"))
    df = build_dataset(audio_paths)
    return df

In [6]:
machine_dfs = get_data_rnn('raw_data/0db_fan')
data_ori = get_dataframe(machine_dfs, 'fan')
# then you get the dataframe of the fan:
data_ori

dict_keys(['fan'])


,noise,machine,id,label,file,snr,f_000,f_001,f_002,f_003,...,f_090,f_091,f_092,f_093,f_094,f_095,f_096,f_097,f_098,f_099
0,0_dB,fan,id_00,abnormal,00000059.wav,0,0.008489,0.009500,0.010201,0.009725,...,0.009003,0.008483,0.009588,0.008255,0.009919,0.009530,0.010291,0.009986,0.009617,0.009319
1,0_dB,fan,id_00,abnormal,00000071.wav,0,0.008590,0.009991,0.008901,0.009887,...,0.008706,0.010670,0.012025,0.010004,0.010868,0.010346,0.008103,0.009203,0.008105,0.007748
2,0_dB,fan,id_00,abnormal,00000065.wav,0,0.010086,0.009601,0.009174,0.009319,...,0.010762,0.009549,0.008739,0.009780,0.009307,0.010024,0.008293,0.008500,0.008613,0.008502
3,0_dB,fan,id_00,abnormal,00000273.wav,0,0.006436,0.007534,0.006199,0.006542,...,0.005720,0.005822,0.004587,0.006939,0.006027,0.004920,0.005144,0.005650,0.005638,0.005438
4,0_dB,fan,id_00,abnormal,00000267.wav,0,0.004283,0.003697,0.004179,0.004582,...,0.009278,0.009251,0.008619,0.007901,0.006953,0.006067,0.005918,0.005611,0.005764,0.004165
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5545,0_dB,fan,id_02,normal,00000068.wav,0,0.007421,0.007069,0.008072,0.008831,...,0.006115,0.006542,0.006682,0.006325,0.006270,0.006890,0.006674,0.007352,0.007334,0.006161
5546,0_dB,fan,id_02,normal,00000040.wav,0,0.005930,0.006486,0.006481,0.006249,...,0.006547,0.007372,0.006891,0.006323,0.006796,0.007167,0.006789,0.006511,0.006546,0.006452
5547,0_dB,fan,id_02,normal,00000726.wav,0,0.006656,0.006478,0.007011,0.006449,...,0.006089,0.005552,0.006399,0.006274,0.006909,0.006791,0.005945,0.006180,0.005910,0.006244
5548,0_dB,fan,id_02,normal,00000732.wav,0,0.006282,0.006609,0.005884,0.006572,...,0.005880,0.006130,0.007057,0.006705,0.007023,0.006236,0.006337,0.006249,0.005911,0.006746


In [ ]:
# save the dataframe to .csv
data_ori.to_csv("aerosound/model/0db_fan_std.csv", index=False)